# 01 — Demographics & Population
**Source:** India Districts Census 2011 · 640 districts  
Covers population totals, gender ratio, age groups, and SC/ST representation.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population, f.male, f.female,
           f.sc, f.st,
           f.age_group_0_29, f.age_group_30_49, f.age_group_50,
           f.rural_households, f.urban_households, f.households
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)

conn.close()

# Derived columns
df['sex_ratio'] = (df['female'] / df['male'] * 1000).round(1)
df['sc_pct']    = (df['sc'] / df['population'] * 100).round(2)
df['st_pct']    = (df['st'] / df['population'] * 100).round(2)
df['youth_pct'] = (df['age_group_0_29'] / df['population'] * 100).round(2)
df['middle_pct']= (df['age_group_30_49'] / df['population'] * 100).round(2)
df['senior_pct']= (df['age_group_50'] / df['population'] * 100).round(2)

state = df.groupby('state_name').agg(
    population=('population','sum'),
    male=('male','sum'),
    female=('female','sum'),
    sc=('sc','sum'),
    st=('st','sum'),
    age_0_29=('age_group_0_29','sum'),
    age_30_49=('age_group_30_49','sum'),
    age_50=('age_group_50','sum'),
).reset_index()

state['sex_ratio'] = (state['female'] / state['male'] * 1000).round(1)
state['sc_pct']    = (state['sc'] / state['population'] * 100).round(2)
state['st_pct']    = (state['st'] / state['population'] * 100).round(2)
state['youth_pct'] = (state['age_0_29'] / state['population'] * 100).round(2)

print(f'Districts: {len(df)}  |  States/UTs: {df.state_name.nunique()}')
print(f'Total Population: {df.population.sum():,.0f}')

## 1.1 — Population by State (Treemap)

In [ ]:
fig = px.treemap(
    state.sort_values('population', ascending=False),
    path=['state_name'],
    values='population',
    color='population',
    color_continuous_scale='Blues',
    title='Population Distribution Across Indian States (2011 Census)',
    hover_data={'population': ':,'},
)
fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))
fig.show()

## 1.2 — Top 15 Most Populous States (Bar Chart)

In [ ]:
top15 = state.nlargest(15, 'population')

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top15['state_name'], top15['population'] / 1e6, color=sns.color_palette('Blues_r', 15))
ax.set_xlabel('Population (Millions)')
ax.set_title('Top 15 Most Populous States — 2011 Census', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, top15['population']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.1f}M', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 1.3 — Gender (Sex) Ratio by State (Females per 1000 Males)

In [ ]:
state_sr = state.sort_values('sex_ratio', ascending=True)

colors = ['#e74c3c' if r < 950 else '#2ecc71' if r >= 1000 else '#3498db' for r in state_sr['sex_ratio']]

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(state_sr['state_name'], state_sr['sex_ratio'], color=colors)
ax.axvline(1000, color='black', linestyle='--', linewidth=1, label='Parity (1000)')
ax.set_xlabel('Sex Ratio (Females per 1000 Males)')
ax.set_title('Sex Ratio by State — 2011 Census\n(Red < 950 | Blue 950–1000 | Green ≥ 1000)', fontsize=13, fontweight='bold')
ax.legend()
for bar, val in zip(bars, state_sr['sex_ratio']):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 1.4 — Age Group Distribution by State (Stacked Bar)

In [ ]:
age_data = state.copy()
age_data['youth_pct']  = age_data['age_0_29'] / age_data['population'] * 100
age_data['middle_pct'] = age_data['age_30_49'] / age_data['population'] * 100
age_data['senior_pct'] = age_data['age_50'] / age_data['population'] * 100
age_data = age_data.sort_values('youth_pct', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='0–29 (Youth)',   x=age_data['state_name'], y=age_data['youth_pct'],  marker_color='#3498db'))
fig.add_trace(go.Bar(name='30–49 (Middle)', x=age_data['state_name'], y=age_data['middle_pct'], marker_color='#e67e22'))
fig.add_trace(go.Bar(name='50+ (Senior)',   x=age_data['state_name'], y=age_data['senior_pct'], marker_color='#9b59b6'))

fig.update_layout(
    barmode='stack',
    title='Age Group Distribution by State (%) — sorted by Youth Share',
    xaxis_tickangle=-45,
    yaxis_title='% of Population',
    height=500,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 1.5 — SC & ST Population Share by State (Scatter Plot)

In [ ]:
fig = px.scatter(
    state,
    x='sc_pct', y='st_pct',
    size='population', color='state_name',
    hover_name='state_name',
    labels={'sc_pct': 'Scheduled Caste % of Population', 'st_pct': 'Scheduled Tribe % of Population'},
    title='SC vs ST Population Share by State (bubble size = total population)',
    size_max=60
)
fig.update_layout(showlegend=False, height=550)
fig.show()

## 1.6 — District-Level Sex Ratio Distribution (Histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df['sex_ratio'].dropna(), bins=40, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(df['sex_ratio'].mean(), color='red', linestyle='--', label=f"Mean: {df['sex_ratio'].mean():.0f}")
ax.axvline(1000, color='green', linestyle='--', label='Parity: 1000')
ax.set_xlabel('Sex Ratio (Females per 1000 Males)')
ax.set_ylabel('Number of Districts')
ax.set_title('Distribution of Sex Ratio Across 640 Districts', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 1.7 — Top 10 Districts with Highest SC Concentration

In [ ]:
top_sc = df.nlargest(10, 'sc_pct')[['district_name', 'state_name', 'sc_pct']]
top_st = df.nlargest(10, 'st_pct')[['district_name', 'state_name', 'st_pct']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top_sc['district_name'] + ' (' + top_sc['state_name'] + ')', top_sc['sc_pct'],
         color=sns.color_palette('Purples_r', 10))
ax1.set_xlabel('SC % of Population')
ax1.set_title('Top 10 Districts by SC Share', fontweight='bold')
ax1.invert_yaxis()

ax2.barh(top_st['district_name'] + ' (' + top_st['state_name'] + ')', top_st['st_pct'],
         color=sns.color_palette('Oranges_r', 10))
ax2.set_xlabel('ST % of Population')
ax2.set_title('Top 10 Districts by ST Share', fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()